# Assignment-12: Custom MLP for Imbalanced Tabular Classification
## Problem Statement: Predicting Credit Card Defaults

In this assignment, you will step away from clean, balanced computer vision datasets (like MNIST) and tackle a real-world financial dataset: the **UCI Default of Credit Card Clients Dataset**. The dataset is uploaded in the same git folder for your reference.

Because this dataset exhibits significant class imbalance (~22% defaults vs ~78% non-defaults), standard training and evaluation routines will fail. You must implement advanced data management, specialized loss scaling, and detailed metric tracking using PyTorch's core architecture.

### Objectives:
1. Handle categorical, numerical, and imbalanced data in a custom PyTorch pipeline.
2. Build a modular MLP architecture using `nn.Module`, initializing weights appropriately.
3. Incorporate a weighted loss function to combat class imbalance.
4. Implement diagnostic checks to monitor gradient health (vanishing gradients) during training.
5. Evaluate your model using metrics robust to severe class imbalance.

### Steps to solve and upload the assignment 

- Download the notebook in your local machine.
- Solve the questions in the notebook and save it.
- Rename the file as `Assignment-12-<your_name>_<your_surname>.ipynb`. For example if your name is Dipika Chopra then name the file as `Assignment-12-Dipika_Chopra.ipynb`.
- Upload the solved notebook to your respective github repository.
- Upload the solved notebook and the scanned pdf copy of the solutions in the google drive location: https://drive.google.com/drive/folders/1WyYyfoJxpvHUZANqXyksOW3FSSsEN-Z9?usp=sharing

## Step 1: Preprocessing & Custom Stratified Datasets

Load your dataset using Pandas. Note that the target column is `default.payment.next.month`. 

**Your Tasks:**
1. **Feature Separation:** Clean the dataset by removing the `ID` column. Identify which features are continuous (e.g., `LIMIT_BAL`, `BILL_AMT1`) and which are categorical/ordinal (e.g., `SEX`, `EDUCATION`, `MARRIAGE`, `PAY_0`).
2. **Feature Scaling:** Normalize continuous variables using `StandardScaler`. 
3. **Stratified Train/Val/Test Split:** Split the data into 80% Train, 10% Validation, and 10% Test. Crucially, ensure the split is **stratified** based on the target class.
4. **PyTorch Pipeline:** Build a custom `Dataset` class overriding `__init__`, `__len__`, and `__getitem__`. Construct training, validation, and test `DataLoader` instances with an appropriate batch size.

---
### Intelligent Design Questions
* **Question 1.1:** Why is a standard random train/test split dangerous for highly imbalanced datasets? What specific problem could manifest in our 10% validation or test sets if stratification is omitted?
* **Question 1.2:** Why must we fit the `StandardScaler` *only* on the training partition instead of scaling the entire raw dataset before the split?

## Step 2: Custom MLP Architecture with Modular Control

Create an explicit network subclassing `nn.Module`. Avoid wrapping your entire model within a single massive `nn.Sequential` block; instead, define layers as standalone structural parameters inside `__init__` and pass inputs explicitly through them within the `forward` function.

**Your Architecture Requirements:**
* At least 3 hidden layers.
* Apply **Kaiming (He) Uniform Initialization** explicitly to the weights of your linear layers, and set the biases to zero.
* Use `nn.ReLU` activations for hidden features.
* Include structural regularization using `nn.Dropout` and stabilization via `nn.BatchNorm1d`.

---
### Intelligent Design Questions
* **Question 2.1:** What is the specific mathematical danger of initializing all weights to zero in a deep neural network? How does He Initialization solve this?
* **Question 2.2:** What is the standard recommended operational order for a single hidden block containing a Linear Layer, Batch Normalization, Dropout, and a ReLU Activation? Justify the placement of Batch Normalization relative to the activation function.

## Step 3: Weighted Loss, Optimization & Gradient Health Diagnostics

Standard Binary Cross-Entropy treats errors on positive (default) and negative (non-default) examples equally. You will replace it with a weighted variation and construct a training loop that tracks gradient health.

**Your Tasks:**
1. **Loss Selection:** Instantiate `nn.BCEWithLogitsLoss`. Compute and provide the precise `pos_weight` factor required to penalize mistakes on the minority default class more severely.
2. **Optimizer:** Use the `Adam` optimizer with a well-chosen learning rate.
3. **Training Loop:** Write an explicit loop tracking training and validation loss across epochs. 
4. **Early Stopping:** Program a safety condition to halt training if validation loss does not drop for 5 consecutive epochs.
5. **Gradient Inspection Hook:** Inside your training loop (immediately after executing `loss.backward()`), print or log the average absolute gradient magnitude (`.grad.abs().mean()`) of the *first* hidden layer versus the *final* hidden layer every few epochs.

---
### Intelligent Design Questions
* **Question 3.1:** Write down the mathematical formula you used to calculate the `pos_weight` value for this specific dataset.
* **Question 3.2:** `nn.BCEWithLogitsLoss` combines a Sigmoid activation and Binary Cross Entropy into one class. Why does PyTorch highly recommend using this unified layer instead of combining a separate `nn.Sigmoid` output layer with a standalone `nn.BCELoss`? 
* **Question 3.3:** Based on your gradient logging task, how can you empirically prove whether or not your deep network is suffering from a **Vanishing Gradient** problem in its early layers as training progresses?

## Step 4: Robust Imbalance Evaluation & Model Saving

Accuracy is entirely uninformative for this dataset. You must evaluate your model's real-world financial utility using tools designed for class imbalance.

**Your Tasks:**
1. Switch your network into evaluation mode.
2. Generate predictions on your completely unseen Test Set. 
3. Calculate and display: **Precision**, **Recall**, **F1-Score**, and a **Confusion Matrix**.
4. Save the model's parameters safely to disk using its `state_dict`.

---
### Intelligent Design Questions
* **Question 4.1:** From a risk management perspective for a bank, which error is worse: a **False Positive** (predicting a client will default when they won't) or a **False Negative** (predicting a client won't default when they actually will)? Which evaluation metric (Precision or Recall) directly tracks this worst-case error?
* **Question 4.2:** What programmatic adjustments must you make to the model's raw output probabilities if the bank demands that you capture at least 80% of all potential defaults, even if it reduces precision?
* **Question 4.3:** Explain why saving the network using `model.state_dict()` is considered a better, more robust software engineering choice than using `torch.save(model)`.